In [ ]:
!pip install opencv-python ipyevents

import cv2
import ipywidgets as widgets
from IPython.display import display
import PIL.Image
import io
import json
import os

OUTPUT_FOLDER = "/kaggle/working/annotations"
# create output dir if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

THETIS_CLASSES = [
    "backhand2hands", "backhand", "backhand_slice", "backhand_volley",
    "forehand_flat", "forehand_openstands", "forehand_slice", "forehand_volley",
    "flat_service", "kick_service", "slice_service", "smash"
]

MAX_DISPLAY_WIDTH = '1200px'

TARGET_FPS = 24

In [2]:
class VideoAnnotator:
    def __init__(self, video_path, output_path, max_width='100%'):
        self.video_path = video_path
        self.output_path = output_path
        self.cap = cv2.VideoCapture(video_path)
        
        # --- FPS Calculation ---
        self.orig_fps = self.cap.get(cv2.CAP_PROP_FPS)
        if self.orig_fps == 0 or self.orig_fps is None: 
            self.orig_fps = 24.0 
        self.fps_ratio = self.orig_fps / TARGET_FPS
        
        self.physical_total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self.total_logical_frames = int(self.physical_total_frames / self.fps_ratio)
        self.current_logical_frame = 0
        self.annotations = []
        
        if os.path.exists(output_path):
            with open(output_path, 'r') as f:
                try:
                    self.annotations = json.load(f)
                except json.JSONDecodeError:
                    self.annotations = []

        # --- UI Components ---
        self.image_widget = widgets.Image(
            format='jpeg',
            layout=widgets.Layout(max_width=max_width, width='100%')
        )
        
        self.btn_prev = widgets.Button(description='< Prev', icon='arrow-left')
        self.btn_next = widgets.Button(description='Next >', icon='arrow-right')
        
        # --- KEYBOARD CONTROL BOX ---
        # We give it a unique class so we can target it with CSS
        self.command_box = widgets.Text(
            value='',
            description='Key Control:',
            layout=widgets.Layout(width='200px')
        )
        self.command_box.add_class('transparent-input')
        
        self.frame_slider = widgets.IntSlider(
            value=0, min=0, max=self.total_logical_frames-1, 
            description='Frame (24fps):', continuous_update=False,
            layout=widgets.Layout(width='100%', max_width=max_width)
        )
        
        self.dropdown = widgets.Dropdown(
            options=['Select Shot...'] + THETIS_CLASSES,
            value='Select Shot...',
            description='Log Shot:',
        )
        
        self.next_player_label = widgets.Label(value=self.get_next_player_text())
        self.status_label = widgets.Label(value=f"Loaded {len(self.annotations)} annotations.")

        self.btn_undo = widgets.Button(
            description='Undo Last', button_style='warning', icon='undo'
        )

        # --- Event Bindings ---
        self.btn_prev.on_click(self.on_prev)
        self.btn_next.on_click(self.on_next)
        self.btn_undo.on_click(self.on_undo)
        self.frame_slider.observe(self.on_slider_change, names='value')
        self.dropdown.observe(self.on_annotate, names='value')
        self.command_box.observe(self.on_command_input, names='value')

        # --- Layout ---
        self.nav_controls = widgets.HBox([self.btn_prev, self.btn_next, self.command_box])
        self.anno_controls = widgets.HBox([self.dropdown, self.next_player_label])
        
        # CSS to hide text and cursor in the specific input box
        css = widgets.HTML("""
        <style>
        .transparent-input input {
            color: transparent !important;
            caret-color: transparent !important;
            font-weight: bold; /* Just in case */
        }
        /* Optional: Hide the background focus border to make it even more subtle */
        .transparent-input input:focus {
            box-shadow: none !important;
            border-color: #cccccc !important;
        }
        </style>
        """)

        self.ui = widgets.VBox([
            css,
            self.image_widget,
            self.frame_slider,
            self.nav_controls,
            widgets.HTML("<hr>"),
            self.anno_controls,
            self.btn_undo,
            self.status_label
        ])
        
        self.update_display()

    def get_next_player(self):
        return 1 if len(self.annotations) % 2 == 0 else 2

    def get_next_player_text(self):
        return f"Next: Player {self.get_next_player()}"

    def update_display(self):
        physical_frame = int(self.current_logical_frame * self.fps_ratio)
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, physical_frame)
        ret, frame = self.cap.read()
        
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = PIL.Image.fromarray(frame)
            f = io.BytesIO()
            img.save(f, format='jpeg')
            self.image_widget.value = f.getvalue()
            self.frame_slider.value = self.current_logical_frame
        else:
            self.status_label.value = "Error reading frame."

    def save_json(self):
        with open(self.output_path, 'w') as f:
            json.dump(self.annotations, f, indent=4)

    def on_prev(self, b):
        if self.current_logical_frame > 0:
            self.current_logical_frame -= 1
            self.update_display()

    def on_next(self, b):
        if self.current_logical_frame < self.total_logical_frames - 1:
            self.current_logical_frame += 1
            self.update_display()

    def on_command_input(self, change):
        text = change['new']
        if not text: return
        
        char = text[-1].lower()
        
        if char == 'e':
            self.on_next(None)
        elif char == 'q':
            self.on_prev(None)
            
        self.command_box.value = ''

    def on_slider_change(self, change):
        self.current_logical_frame = change['new']
        self.update_display()

    def on_undo(self, b):
        if self.annotations:
            removed = self.annotations.pop()
            self.save_json()
            self.status_label.value = f"Removed last entry: {removed['shot']}"
            self.next_player_label.value = self.get_next_player_text()

    def on_annotate(self, change):
        shot = change['new']
        if shot == 'Select Shot...': return

        player = self.get_next_player()
        record = {
            "frame": self.current_logical_frame,
            "shot": shot,
            "player": player
        }
        self.annotations.append(record)
        self.save_json()
        self.status_label.value = f"Saved: {shot} (P{player}) at frame {self.current_logical_frame}"
        self.next_player_label.value = self.get_next_player_text()
        self.dropdown.value = 'Select Shot...'

    def show(self):
        display(self.ui)

In [3]:
#IMPORTANT STUFF
VIDEO_PATH = '/kaggle/input/tennis-rally-videos/input_video.mp4'
OUTPUT_JSON = '/kaggle/working/annotations/input_video.json'


# --- Run ---
annotator = VideoAnnotator(VIDEO_PATH, OUTPUT_JSON, max_width=MAX_DISPLAY_WIDTH)
annotator.show()